# Notebook 3: AI Agents and Agent-to-Agent (A2A) Evaluation

This notebook demonstrates AI agents for portfolio optimization with **automatic Langfuse tracing** via `@observe()` decorator.

1. **Portfolio Optimization Agent** - LangChain agent with portfolio tools
2. **Evaluator Agent** - Agent with RAG for evaluation
3. **Agent-to-Agent (A2A) Protocol** - Agents communicate and evaluate each other

**Key**: Just use `@observe()` decorator on functions - Langfuse traces everything automatically including nested LangChain tool calls.

## Setup

In [1]:
import os
import json
import warnings
from typing import Dict, Any, List, Optional
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
load_dotenv(dotenv_path='../.env')

# Configure Gemini API
import google.generativeai as genai
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if gemini_api_key:
    genai.configure(api_key=gemini_api_key)
    os.environ["GOOGLE_API_KEY"] = gemini_api_key

from langchain.agents import AgentExecutor

# Import agents - Langfuse tracing via @observe() decorator
from agents import (
    create_portfolio_agent, create_evaluator_agent, create_rag_knowledge_base,
    run_portfolio_agent, run_a2a_evaluation, A2AEvaluation,
    PORTFOLIO_TOOLS, flush_langfuse, LLMProvider
)
from portfolio_optimizer import UNIVERSE_DEFINITIONS

# Load data from JSON files
with open('scenarios.json', 'r') as f:
    SCENARIOS = json.load(f)
with open('evaluation_dataset.json', 'r') as f:
    EVAL_DATA = json.load(f)

# Default provider for this notebook (Gemini)
DEFAULT_PROVIDER = LLMProvider.GEMINI

warnings.filterwarnings('ignore')
print(f"Gemini API configured: {'Yes' if gemini_api_key else 'No'}")
print(f"Default LLM Provider: {DEFAULT_PROVIDER.value}")
print(f"Loaded {len(SCENARIOS['personas'])} personas, {len(EVAL_DATA['items'])} eval items")
print("\nLangfuse: @observe() decorator traces everything automatically")
print("Setup complete!")

Gemini API configured: Yes
Default LLM Provider: gemini
Loaded 3 personas, 20 eval items

Langfuse: @observe() decorator traces everything automatically
Setup complete!


## 1. Portfolio Agent with Tools

The Portfolio Agent has four tools:
- `get_available_universes` - List available asset universes
- `optimize_portfolio_tool` - Run MVO optimization
- `run_hrp_optimization` - Run HRP optimization
- `compare_portfolios` - Compare optimization methods

**All tool calls traced inside the agent via Langfuse CallbackHandler.**

In [2]:
# Show available tools
print(f"Portfolio Agent Tools ({len(PORTFOLIO_TOOLS)}):")
for tool in PORTFOLIO_TOOLS:
    print(f"  - {tool.name}: {tool.description[:50]}...")

Portfolio Agent Tools (4):
  - get_available_universes: Get available asset universes for portfolio constr...
  - optimize_portfolio_tool: Optimize a portfolio using Mean-Variance Optimizat...
  - run_hrp_optimization: Run Hierarchical Risk Parity optimization....
  - compare_portfolios: Compare different optimization approaches for the ...


In [3]:
# Create portfolio agent (using default provider - Gemini)
portfolio_agent = create_portfolio_agent(provider=DEFAULT_PROVIDER)
print(f"Portfolio agent created with {DEFAULT_PROVIDER.value}!")

Key 'title' is not supported in schema, ignoring


Key 'title' is not supported in schema, ignoring


Key 'title' is not supported in schema, ignoring


Key 'title' is not supported in schema, ignoring


Key 'default' is not supported in schema, ignoring


...

Key 'title' is not supported in schema, ignoring


Key 'title' is not supported in schema, ignoring


Key 'title' is not supported in schema, ignoring


Key 'title' is not supported in schema, ignoring


Portfolio agent created with gemini!


In [4]:
# Test portfolio agent with Elena's persona
elena = SCENARIOS['personas']['elena_balanced']
print(f"Testing Portfolio Agent with {elena['name']}'s persona...\n")

# run_portfolio_agent uses @observe() - automatic tracing
result = run_portfolio_agent(portfolio_agent, elena['narrative'], session_id="notebook3_demo")

print("\n" + "="*60)
print("Agent Response (truncated):")
print(result["output"][:500] + "..." if len(result["output"]) > 500 else result["output"])
print("\n" + "="*60)
print(f"Tools used: {len(result.get('intermediate_steps', []))} tool calls")
print("\nIn Langfuse: portfolio_agent trace with nested tool calls")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Testing Portfolio Agent with Elena Rodriguez's persona...



I0000 00:00:1768478044.888540 29510874 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



Invoking: `get_available_universes` with `{}`
responded: Okay Elena, I can help you construct a portfolio for your children's education fund. Given your 18-year time horizon and moderate risk tolerance, a globally diversified portfolio using Mean-Variance Optimization seems appropriate. I will set a maximum position size of 15% as you requested.

First, let's check the available universes:


{
  "us_large_cap": [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META",
    "NVDA",
    "TSLA",
    "BRK-B",
    "JPM",
    "JNJ",
    "V",
    "UNH",
    "HD",
    "PG",
    "MA"
  ],
  "us_tech": [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META",
    "NVDA",
    "TSLA",
    "CRM",
    "ADBE",
    "INTC",
    "CSCO",
    "ORCL",
    "IBM",
    "QCOM",
    "AMD"
  ],
  "global_diversified": [
    "SPY",
    "EFA",
    "EEM",
    "VWO",
    "TLT",
    "GLD",
    "VNQ",
    "LQD",
    "HYG",
    "DBC",
    "IEF",
    "GOVT",
    "AGG",
    "BND",
    "VTI"
  ],
  "european": [
 


Invoking: `optimize_portfolio_tool` with `{'max_position': 0.15, 'universe': 'global_diversified', 'optimization_target': 'max_sharpe'}`
responded: Okay, Elena. Given your investment horizon of 18 years, moderate risk tolerance, and desire for global diversification, the "global_diversified" universe seems most suitable. It includes a mix of US stocks (SPY, VTI), developed international stocks (EFA), emerging market stocks (EEM, VWO), bonds (TLT, LQD, HYG, IEF, GOVT, AGG, BND), gold (GLD) and real estate (VNQ), and commodities (DBC) providing broad diversification.

I will use Mean-Variance Optimization with the goal of maximizing the Sharpe Ratio, which balances risk and return. I will also set a maximum position size of 15% as you requested.




{
  "universe": "global_diversified",
  "optimization_target": "max_sharpe",
  "weights": {
    "AGG": 0.15,
    "BND": 0.0419,
    "DBC": 0.15,
    "GLD": 0.15,
    "GOVT": 0.15,
    "IEF": 0.0581,
    "SPY": 0.15,
    "VTI": 0.15
  },
  "expected_return": 7.84,
  "volatility": 9.25,
  "sharpe_ratio": 0.631,
  "backtest_sharpe": 0.932,
  "max_drawdown": -15.4
}

Okay, Elena, let's build a portfolio for your children's education fund. Given your 18-year time horizon, moderate risk tolerance, and desire for global diversification, I recommend using the "global_diversified" universe. I will use Mean-Variance Optimization with the goal of maximizing the Sharpe Ratio, which balances risk and return. I will also set a maximum position size of 15% to ensure diversification.

Here's the portfolio I've created:

*   **Universe:** global\_diversified
*   **Optimization Target:** max\_sharpe
*   **Maximum Position:** 15%

**Portfolio Composition:**

*   AGG: 15.00%
*   BND: 4.19%
*   DBC: 15.00%
*   GLD: 15.00%
*   GOVT: 15.00%
*   IEF: 5.81%
*   SPY: 15.00%
*   VTI: 15.00%

**Backtest Results:**

*   **Sharpe Ratio:** 0.631
*   **Expected Return:** 7.84%
*   **Volatility:** 9.25%
*   **Max Drawdown:** -15.4%

This portfolio provides broad diversification across global equities (SPY, VTI), developed international equities (EFA), emerging market equities 

## 2. Evaluator Agent with RAG

The Evaluator Agent has:
- `search_knowledge_base` - RAG retrieval from evaluation dataset

**Tool calls traced inside the agent via Langfuse CallbackHandler.**

In [5]:
# Create RAG knowledge base
retriever = create_rag_knowledge_base(EVAL_DATA)
print(f"RAG knowledge base created")
print(f"Personas included: {', '.join([p['persona'] for p in EVAL_DATA['rag_knowledge']])}")

RAG knowledge base created
Personas included: Sarah Chen, Marcus Johnson, Elena Rodriguez


In [6]:
# Create evaluator agent with RAG (using default provider - Gemini)
evaluator_agent = create_evaluator_agent(retriever, provider=DEFAULT_PROVIDER)
print(f"Evaluator agent created with {DEFAULT_PROVIDER.value} and RAG tool!")

Key 'title' is not supported in schema, ignoring


Key 'title' is not supported in schema, ignoring


Evaluator agent created with gemini and RAG tool!


## 3. Agent-to-Agent (A2A) Evaluation

The A2A protocol runs two agents:
1. **Portfolio Agent** - Handles the investor request
2. **Evaluator Agent** - Evaluates the portfolio agent's response using RAG

**Tracing is automatic** - the `@observe()` decorator on `run_a2a_evaluation()` creates a trace, and `CallbackHandler()` inside inherits the context:

```
a2a_evaluation (trace)
├── Portfolio Agent chain
│   ├── LLM call
│   ├── get_available_universes (tool)
│   └── optimize_portfolio_tool (tool)
└── Evaluator Agent chain
    ├── LLM call
    ├── search_knowledge_base (tool)
    └── LLM call (final evaluation)
```

In [7]:
# Run A2A evaluation on Elena's persona
elena = SCENARIOS['personas']['elena_balanced']

print(f"Running A2A Evaluation on {elena['name']}'s persona...\n")

# A2A evaluation creates two agent traces with nested tool calls
a2a_result = run_a2a_evaluation(
    task_description=elena['narrative'],
    portfolio_agent=portfolio_agent,
    evaluator_agent=evaluator_agent,
    session_id="notebook3_a2a"
)

print("\n" + "="*60)
print("A2A EVALUATION RESULTS")
print("="*60)
print(f"\nOverall Score: {a2a_result.overall_score:.1f}/10")
print("\nDimension Scores:")
for dim, score in a2a_result.scores.items():
    print(f"  {dim}: {score:.1f}/10")

print("\n" + "="*60)
print("LANGFUSE TRACES CREATED:")
print("  - a2a_portfolio_agent: Portfolio agent with nested tool calls")
print("  - a2a_evaluator_agent: Evaluator agent with RAG tool calls")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Running A2A Evaluation on Elena Rodriguez's persona...




Invoking: `get_available_universes` with `{}`
responded: Okay Elena, I can help you construct a portfolio for your children's education fund. Given your 18-year time horizon and moderate risk tolerance, a globally diversified portfolio using Mean-Variance Optimization seems appropriate. I will set a maximum position size of 15% as you requested.

First, let's check the available universes:


{
  "us_large_cap": [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META",
    "NVDA",
    "TSLA",
    "BRK-B",
    "JPM",
    "JNJ",
    "V",
    "UNH",
    "HD",
    "PG",
    "MA"
  ],
  "us_tech": [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META",
    "NVDA",
    "TSLA",
    "CRM",
    "ADBE",
    "INTC",
    "CSCO",
    "ORCL",
    "IBM",
    "QCOM",
    "AMD"
  ],
  "global_diversified": [
    "SPY",
    "EFA",
    "EEM",
    "VWO",
    "TLT",
    "GLD",
    "VNQ",
    "LQD",
    "HYG",
    "DBC",
    "IEF",
    "GOVT",
    "AGG",
    "BND",
    "VTI"
  ],
  "european": [
 


Invoking: `optimize_portfolio_tool` with `{'max_position': 0.15, 'universe': 'global_diversified', 'optimization_target': 'max_sharpe'}`
responded: Okay, Elena. Given your investment goals, moderate risk tolerance, and long time horizon (18 years), a globally diversified portfolio seems appropriate. I see several available universes, and the "global_diversified" universe aligns well with your needs. It includes a mix of US stocks (SPY, VTI), developed international stocks (EFA), emerging market stocks (EEM, VWO), bonds (TLT, LQD, HYG, IEF, GOVT, AGG, BND), commodities (DBC), and real estate (VNQ), providing broad diversification.

I will use Mean-Variance Optimization with the "global_diversified" universe and set a maximum position size of 15% to limit concentration risk. Since you're aiming for reasonable risk-adjusted returns, I'll optimize for the maximum Sharpe ratio. This will attempt to maximize your return per unit of risk.




{
  "universe": "global_diversified",
  "optimization_target": "max_sharpe",
  "weights": {
    "AGG": 0.15,
    "BND": 0.0419,
    "DBC": 0.15,
    "GLD": 0.15,
    "GOVT": 0.15,
    "IEF": 0.0581,
    "SPY": 0.15,
    "VTI": 0.15
  },
  "expected_return": 7.84,
  "volatility": 9.25,
  "sharpe_ratio": 0.631,
  "backtest_sharpe": 0.932,
  "max_drawdown": -15.4
}

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Okay, Elena, let's build a portfolio for your children's education fund. Given your 18-year time horizon and moderate risk tolerance, a globally diversified portfolio makes sense. I will use the "global_diversified" universe. I will aim to maximize the Sharpe ratio, which balances risk and return. I will also limit any single position to 15% of the portfolio as you requested.

Here's the portfolio I've constructed:

*   **Universe:** global\_diversified
*   **Optimization Target:** max\_sharpe
*   **Maximum Position:** 15%

**Portfolio Composition:**

*   AGG: 15.00%
*   BND: 4.19%
*   DBC: 15.00%
*   GLD: 15.00%
*   GOVT: 15.00%
*   IEF: 5.81%
*   SPY: 15.00%
*   VTI: 15.00%

**Backtesting Results:**

*   **Backtest Sharpe Ratio:** 0.932
*   **Expected Return:** 7.84%
*   **Volatility:** 9.25%
*   **Sharpe Ratio:** 0.631
*   **Max Drawdown:** -15.4%

This portfolio provides global diversification across stocks (SPY, VTI), bonds (AGG, BND, GOVT, IEF), commodities (DBC), and gold (GLD).

Okay, I will evaluate the portfolio recommendation based on the criteria you provided.

**Scores:**

*   **Universe Selection:** 8/10
*   **Optimization Method:** 7/10
*   **Risk Assessment:** 7/10
*   **Constraint Handling:** 10/10
*   **Explanation Quality:** 8/10
*   **Overall Score:** 8/10

**Feedback:**

*   **Universe Selection:** The "global\_diversified" universe is a reasonable starting point for Elena's goals. It aligns with her desire for global diversification. However, it would be helpful to know the specific assets included in this universe to better assess its suitability. A more tailored universe, perhaps focusing on specific ETFs or asset classes relevant to long-term growth and education savings, could potentially improve the portfolio.

*   **Optimization Method:** Maximizing the Sharpe ratio is a common and generally sound approach. However, for a long-term goal like education savings, other optimization targets might be considered, such as maximizing after-tax retu

## 4. Batch A2A Evaluation

Run A2A evaluation on multiple scenarios. Each scenario creates two agent traces.

In [8]:
# Load complex scenarios
COMPLEX_SCENARIOS = SCENARIOS['complex_scenarios']
print(f"Complex scenarios ({len(COMPLEX_SCENARIOS)}):")
for s in COMPLEX_SCENARIOS:
    print(f"  - {s['name']}")

Complex scenarios (3):
  - Multi-objective comparison for Elena
  - Position limit analysis for Marcus
  - Risk strategy for Sarah


In [9]:
def evaluate_complex_scenarios(
    scenarios: List[Dict],
    portfolio_agent: AgentExecutor,
    evaluator_agent: AgentExecutor,
    max_scenarios: int = 2
) -> List[A2AEvaluation]:
    """Evaluate multiple scenarios using A2A protocol.
    
    Each scenario creates:
    - a2a_portfolio_agent trace with nested tool calls
    - a2a_evaluator_agent trace with nested RAG calls
    """
    results = []
    
    for i, scenario in enumerate(scenarios[:max_scenarios]):
        print(f"\n{'#'*60}")
        print(f"SCENARIO {i+1}: {scenario['name']}")
        print(f"{'#'*60}")
        
        # Each A2A evaluation creates two agent traces
        eval_result = run_a2a_evaluation(
            task_description=scenario["description"],
            portfolio_agent=portfolio_agent,
            evaluator_agent=evaluator_agent,
            session_id=f"notebook3_complex_{i}"
        )
        results.append(eval_result)
        
        print(f"\nScore: {eval_result.overall_score:.1f}/10")
    
    return results

print("Evaluating complex scenarios...\n")
complex_results = evaluate_complex_scenarios(
    COMPLEX_SCENARIOS,
    portfolio_agent,
    evaluator_agent,
    max_scenarios=1
)

print("\n" + "="*60)
print("Complex scenario evaluation complete!")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Evaluating complex scenarios...


############################################################
SCENARIO 1: Multi-objective comparison for Elena
############################################################



Invoking: `get_available_universes` with `{}`
responded: Okay, Elena wants to compare three portfolio optimization approaches for her children's education fund:

1.  **Minimum Volatility (Conservative Assets):** This strategy focuses on minimizing risk (volatility) within a conservative asset universe. It's a safe approach, suitable for risk-averse investors.
2.  **Maximum Sharpe Ratio (Global Diversified Assets):** This strategy aims to maximize the Sharpe Ratio, which balances return and risk, using a globally diversified asset universe. Elena prefers this approach.
3.  **Hierarchical Risk Parity (HRP) (Global Diversified Assets):** HRP is an alternative approach that uses hierarchical clustering to diversify risk across assets. It's less sensitive to estimation errors than traditional mean-variance optimization.

I will use the available tools to implement these strategies and then compare the results.

First, I need to know what asset universes are available.


{
  "us_large_cap":


Invoking: `optimize_portfolio_tool` with `{'max_position': 0.15, 'universe': 'conservative', 'optimization_target': 'min_volatility'}`
responded: Okay, Elena wants to compare three portfolio optimization approaches for her children's education fund:

1.  **Minimum Volatility (Conservative Assets):** This aims for the lowest possible risk using the "conservative" universe.
2.  **Maximum Sharpe Ratio (Global Diversified Assets):** This aims for the highest possible risk-adjusted return using the "global\_diversified" universe, which is her preferred approach.
3.  **Hierarchical Risk Parity (Global Diversified Assets):** This is an alternative approach that also uses the "global\_diversified" universe and focuses on diversifying risk across the portfolio.

I will run each optimization with a 15% max position constraint and then present the results.





{
  "universe": "conservative",
  "optimization_target": "min_volatility",
  "weights": {
    "AGG": 0.15,
    "BND": 0.15,
    "GOVT": 0.15,
    "IEF": 0.15,
    "LQD": 0.1,
    "MBB": 0.15,
    "VMBS": 0.15
  },
  "expected_return": 0.72,
  "volatility": 6.4,
  "sharpe_ratio": -0.201,
  "backtest_sharpe": 0.151,
  "max_drawdown": -19.4
}
Invoking: `optimize_portfolio_tool` with `{'max_position': 0.15, 'universe': 'global_diversified', 'optimization_target': 'max_sharpe'}`
responded: Okay, Elena wants to compare three portfolio optimization approaches for her children's education fund:

1.  **Minimum Volatility (Conservative Assets):** This aims for the lowest possible risk using the "conservative" universe.
2.  **Maximum Sharpe Ratio (Global Diversified Assets):** This aims for the highest possible risk-adjusted return using the "global\_diversified" universe, which is her preferred approach.
3.  **Hierarchical Risk Parity (Global Diversified Assets):** This is an alternative approac

...

{
  "universe": "global_diversified",
  "optimization_target": "max_sharpe",
  "weights": {
    "AGG": 0.15,
    "BND": 0.0419,
    "DBC": 0.15,
    "GLD": 0.15,
    "GOVT": 0.15,
    "IEF": 0.0581,
    "SPY": 0.15,
    "VTI": 0.15
  },
  "expected_return": 7.84,
  "volatility": 9.25,
  "sharpe_ratio": 0.631,
  "backtest_sharpe": 0.932,
  "max_drawdown": -15.4
}


Invoking: `optimize_portfolio_tool` with `{'max_position': 0.15, 'universe': 'conservative', 'optimization_target': 'min_volatility'}`


{
  "universe": "conservative",
  "optimization_target": "min_volatility",
  "weights": {
    "AGG": 0.15,
    "BND": 0.15,
    "GOVT": 0.15,
    "IEF": 0.15,
    "LQD": 0.1,
    "MBB": 0.15,
    "VMBS": 0.15
  },
  "expected_return": 0.72,
  "volatility": 6.4,
  "sharpe_ratio": -0.201,
  "backtest_sharpe": 0.151,
  "max_drawdown": -19.4
}

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Okay, Elena, let's analyze these three portfolio approaches for your children's education fund.

**1. Minimum Volatility (Conservative Assets):**

*   **Universe:** Conservative (BND, AGG, TLT, IEF, GOVT, LQD, MBB, VMBS)
*   **Optimization Target:** min\_volatility
*   **Maximum Position:** 15%
*   **Expected Return:** 0.72%
*   **Volatility:** 6.4%
*   **Sharpe Ratio:** -0.201
*   **Max Drawdown:** -19.4%
*   **Weights:** AGG: 15%, BND: 15%, GOVT: 15%, IEF: 15%, LQD: 10%, MBB: 15%, VMBS: 15%

This portfolio prioritizes stability and low risk. The expected return is quite low, but so is the volatility. The negative Sharpe ratio indicates that the risk-free rate is higher than the portfolio's return. The max drawdown suggests a potential loss of almost 20% during adverse market conditions.

**2. Maximum Sharpe Ratio (Global Diversified Assets):**

*   **Universe:** Global Diversified (SPY, EFA, EEM, VWO, TLT, GLD, VNQ, LQD, HYG, DBC, IEF, GOVT, AGG, BND, VTI)
*   **Optimization Target:*

Okay, here's an evaluation of the portfolio recommendation provided, broken down by category:

**1. Universe Selection (Score: 9/10)**

*   **Minimum Volatility:** The universe (BND, AGG, TLT, IEF, GOVT, LQD, MBB, VMBS) is appropriate for a conservative, low-volatility approach, focusing on fixed income.
*   **Maximum Sharpe Ratio & HRP:** The global diversified universe (SPY, EFA, EEM, VWO, TLT, GLD, VNQ, LQD, HYG, DBC, IEF, GOVT, AGG, BND, VTI) is well-suited for both maximum Sharpe ratio and HRP strategies, providing broad exposure to different asset classes and geographies.
*   **Feedback:** The universe selections are generally strong. A minor improvement could be to consider specific sector ETFs within the global diversified universe for even greater granularity, but this is not essential.

**2. Optimization Method (Score: 8/10)**

*   **Minimum Volatility:** Using `min_volatility` is a standard and appropriate choice for a conservative portfolio.
*   **Maximum Sharpe Ratio:** Us

## 5. Key Takeaways

### Simple Tracing with @observe()

**The pattern is simple:**
```python
from langfuse import observe, get_client
from langfuse.langchain import CallbackHandler

@observe()
def run_my_agent(agent, query, session_id=None):
    langfuse = get_client()
    if langfuse:
        langfuse.update_current_trace(name="my_agent", session_id=session_id)
    
    # CallbackHandler auto-inherits current trace context
    handler = CallbackHandler()
    result = agent.invoke({"input": query}, config={"callbacks": [handler]})
    return result
```

**In Langfuse you'll see hierarchical traces:**
```
my_agent (trace)
├── LLM call (generation)
├── tool_call_1 (tool)
├── LLM call (generation)
└── tool_call_2 (tool)
```

### A2A Evaluation
- Single trace contains both agents' execution
- Evaluator uses RAG to provide context-aware assessment
- Scores parsed from evaluator response

In [10]:
# Flush Langfuse events
flush_langfuse()
print("Langfuse events flushed!")
print("\nCheck Langfuse dashboard for:")
print("  - Agent traces with nested tool calls")
print("  - Sessions: notebook3_demo, notebook3_a2a, notebook3_complex")
print("  - Hierarchical view: Agent -> LLM calls -> Tool calls")

Langfuse events flushed!

Check Langfuse dashboard for:
  - Agent traces with nested tool calls
  - Sessions: notebook3_demo, notebook3_a2a, notebook3_complex
  - Hierarchical view: Agent -> LLM calls -> Tool calls
